# 🧠 Memory in LangChain — Explained

**What this notebook does:** builds a conversational RAG (Retrieval-Augmented Generation) chatbot over two research papers (the Transformer paper and the YOLO paper) that **remembers earlier turns of the conversation** — so a follow-up question like "what about YOLO?" is understood in context, not treated as a brand-new, standalone question.

**The flow:**
`install libs → imports & API key → embed PDFs into Chroma → build a history-aware retriever → build the answer-generation chain → combine them into a RAG chain → test it with a manually-managed chat_history list → upgrade to automatic, per-session memory with RunnableWithMessageHistory`

Each code cell below has a plain-English explanation right above it, and the code itself has line-by-line comments.

### 📦 Cell 2 — Install the libraries

Installs everything this notebook needs (run once):
- `langchain-openai` → OpenAI's chat + embedding models for LangChain
- `langchain` → LangChain's core framework
- `pdfminer.six` → extracts raw text out of PDF files
- `chromadb` → the vector database used to store and search embedded document chunks
- `langchain-community` → community-maintained integrations (Chroma wrapper, chat history store)
- `langchain-text-splitters` → chunks long text into overlapping windows before embedding

*(The original version of this notebook used Cohere, which offers free trial API keys from dashboard.cohere.com — this version uses OpenAI instead, so an `OPENAI_API_KEY` is needed instead.)*

In [1]:
# Install every package this notebook depends on (safe to re-run; pip skips what's already installed)
!pip install langchain-openai langchain pdfminer.six chromadb langchain-community langchain-text-splitters

Defaulting to user installation because normal site-packages is not writeable
  Using cached langchain_openai-1.6.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached langchain-1.4.0-py3-none-any.whl.metadata (6.2 kB)
  Using cached chromadb-1.5.9-cp39-abi3-win_amd64.whl.metadata (5.1 kB)
  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached langchain_text_splitters-1.1.2-py3-none-any.whl.metadata (3.3 kB)
  Using cached langchain_core-1.6.3-py3-none-any.whl.metadata (4.8 kB)
  Using cached openai-3.13.0-py3-none-any.whl.metadata (41 kB)
  Using cached tiktoken-0.14.0-cp313-cp313-win_amd64.whl.metadata (6.8 kB)
  Using cached langgraph-1.2.11-py3-none-any.whl.metadata (4.9 kB)
  Using cached build-1.6.1-py3-none-any.whl.metadata (5.6 kB)
  Using cached pybase64-1.5.0-cp313-cp313-win_amd64.whl.metadata (11 kB)
  Using cached opentelemetry_api-1.44.0-py3-none-any.whl.metadata (1.4 kB)
  Using cached opentelemetry_exporter_otlp_proto_grpc-1.44.0-py3-non

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
openai-agents

### 📚 What each library is for

- **langchain-openai**: connects OpenAI's chat and embedding models to LangChain.
- **langchain**: the modular framework for building LLM-powered apps (chatbots, Q&A systems, agents).
- **pdfminer.six**: extracts text from PDF files for downstream processing.
- **chromadb**: a vector database for storing and searching embeddings — the backbone of semantic search here.

### 🔑 Cell 5 — Imports and API key

Sets the OpenAI key and imports every class used later in the notebook.

- The `try/except google.colab.userdata` block: on Google Colab, this reads the key from Colab's secret manager; anywhere else (e.g. running locally), the `import google.colab` fails, we hit `except ImportError`, and we just assume `OPENAI_API_KEY` is already set as an environment variable. This makes the same notebook work on Colab *and* on a local machine.
- `create_history_aware_retriever` → builds a retriever that first rewrites the user's question using chat history (so "what about YOLO?" becomes a standalone question) before searching.
- `create_stuff_documents_chain` → builds the final "answer the question using these retrieved documents" step ("stuff" = all retrieved documents are stuffed directly into the prompt).
- `RunnableWithMessageHistory` → wraps a chain so it automatically loads/saves chat history per session, instead of the caller managing a `chat_history` list by hand.

In [2]:
import os  # standard library: used to read/set environment variables
from typing import List                                       # type hint for list annotations (kept for parity, not directly used below)
from pydantic import BaseModel, Field                          # data-validation base classes (kept for parity, not directly used below)
from langchain_core.messages import BaseMessage, AIMessage, HumanMessage  # message types used to build/read chat history

try:
    from google.colab import userdata  # only importable when running inside Google Colab
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')  # pull the key from Colab's secret manager
except ImportError:
    pass  # not running in Colab -- assume OPENAI_API_KEY is already set in the environment

from langchain_core.prompts import PromptTemplate                        # template for building plain-text prompts
from langchain_community.chat_message_histories import ChatMessageHistory  # in-memory store for a single conversation's messages
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder  # chat-style prompt template + a slot for injecting past messages
from langchain_classic.chains import create_history_aware_retriever, create_retrieval_chain  # builds a chat-history-aware retriever, and the final RAG chain
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI                        # OpenAI chat model wrapper
from langchain_core.output_parsers import StrOutputParser      # parses an LLM response down to a plain string
from pdfminer.high_level import extract_text as extract_text_pdf_miner  # pulls raw text out of a PDF file
from langchain_community.vectorstores import Chroma            # the vector database wrapper (semantic search)
from langchain_openai import OpenAIEmbeddings                   # turns text into embedding vectors via OpenAI's API
from langchain_text_splitters import RecursiveCharacterTextSplitter  # splits long text into overlapping chunks
from langchain_core.documents import Document                   # LangChain's container for a chunk of text + its metadata
from langchain_core.runnables import RunnableParallel, RunnablePassthrough  # LCEL building blocks (kept for parity, not directly used below)
from langchain_core.chat_history import BaseChatMessageHistory   # abstract base class a chat-history store must implement
from langchain_core.runnables.history import RunnableWithMessageHistory  # auto-manages chat history per session for a chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain  # builds the "stuff documents into the prompt, then answer" step

C:\Users\Avado\AppData\Local\Temp\ipykernel_35188\2442543177.py:13: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import ChatMessageHistory


### 📥 Download the source PDFs

The rest of this notebook expects `1706.03762v7.pdf` ("Attention Is All You Need") and `1506.02640v5.pdf` (YOLO) to already exist at `/content/`. On a fresh Colab runtime (or a fresh local run) they won't be there yet, which is what causes a `FileNotFoundError` in the next cells.

This cell fetches both papers directly from arXiv (open-access preprints) and saves them to `/content/`, skipping the download if a file is already present — so it's safe to re-run.

In [ ]:
import os
import urllib.request

os.makedirs("/content", exist_ok=True)  # ensure the folder exists (already there on Colab; needed for a local run)

PAPERS = {
    "/content/1706.03762v7.pdf": "https://arxiv.org/pdf/1706.03762v7",  # "Attention Is All You Need" (Transformer paper)
    "/content/1506.02640v5.pdf": "https://arxiv.org/pdf/1506.02640v5",  # "You Only Look Once" (YOLO paper)
}

for local_path, url in PAPERS.items():
    if not os.path.exists(local_path):    # skip re-downloading if it's already there
        urllib.request.urlretrieve(url, local_path)
        print(f"Downloaded {url} -> {local_path}")
    else:
        print(f"Already present: {local_path}")

## VectorDB setup

### 🧩 Cell 7 — Set up the embedding model + storage location

- `persist_directory` → the folder where Chroma will save its vector index on disk, so it can be reloaded later without re-embedding everything.
- `embedding` → the OpenAI model that turns each text chunk into a vector of numbers ("embedding") that captures its meaning. `text-embedding-3-small` is OpenAI's current low-cost, high-quality embedding model.

In [3]:
# Define the directory where the Chroma database will persist data
persist_directory = "/content/chroma_db"

# Initialize OpenAI embeddings with the specified model
# "text-embedding-3-small" is OpenAI's current, cost-efficient embedding model
embedding = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

### 📄 Cell 9 — Load both PDFs, chunk them, and embed them into Chroma

We're processing two research papers: the Transformer paper (`1706.03762v7.pdf`) and the YOLO paper (`1506.02640v5.pdf`).

- For each PDF: extract raw text with `pdfminer`, collapse newlines into single spaces, then split it into overlapping 2048-character chunks (512-character overlap, so a sentence spanning a chunk boundary isn't lost entirely).
- Each chunk becomes a `Document`, tagged with `source` (the originating file path) as metadata — this is how the chatbot can later cite which paper an answer came from.
- `Chroma.from_documents(...)` embeds every chunk and writes the resulting vectors to disk at `persist_directory`. This runs once per PDF, so by the end both papers' chunks live in the same Chroma collection.

In [4]:
# Loop through a list of PDF files to process
for pdf_name in ["/content/1706.03762v7.pdf", "/content/1506.02640v5.pdf"]:
    # Open each PDF file in binary mode
    with open(pdf_name, 'rb') as f:
        # Extract text from the PDF using the extract_text_pdf_miner function
        text = extract_text_pdf_miner(f)

        # Clean the extracted text by removing newline characters and joining into a single string
        cleaned_text = " ".join(text.split("\n"))

        # Initialize a list to store document chunks
        docs = []

        # Create a text splitter to divide the text into manageable chunks
        # Each chunk has a maximum size of 2048 characters with a 512-character overlap
        splitter = RecursiveCharacterTextSplitter(chunk_size=2048, chunk_overlap=512)

        # Split the cleaned text into chunks and wrap each chunk in a Document object
        for chunk in splitter.split_text(cleaned_text):
            docs.append(Document(page_content=chunk, metadata={"source": pdf_name}))  # tag each chunk with its source PDF

    # Create a Chroma collection from the processed documents
    # Use the specified persist directory and embedding model for storage and retrieval
    vector_collection_fixed_size = Chroma.from_documents(
        documents=docs,               # this PDF's chunks
        persist_directory=persist_directory,  # where to save the vector index
        embedding=embedding            # the embedding model to use
    )

### 🔗 Cell 10 — Reconnect to the Chroma collection

Reloads the persisted Chroma collection built in Cell 9 into a `vectordb` variable, ready to be queried.

In [5]:
# Initialize a Chroma vector database
# The persist_directory specifies the location where the database is stored
# The embedding_function parameter provides the embedding model used for vector representation
vectordb = Chroma(persist_directory=persist_directory, embedding_function=embedding)

C:\Users\Avado\AppData\Local\Temp\ipykernel_35188\1816146847.py:4: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectordb = Chroma(persist_directory=persist_directory, embedding_function=embedding)


### 🔎 Cell 11 — Sanity-check the vector store with a direct similarity search

Before wiring up any chains, this confirms retrieval works end to end: searches for the single (`k=1`) most relevant chunk to `"What is YOLO?"` and returns it along with its relevance score.

In [6]:
# Perform a similarity search on the vector database
# The query "What is YOLO?" is used to find the most relevant documents
# k=1 specifies that the top 1 most similar document should be retrieved
# The method also returns relevance scores indicating how closely each document matches the query
vectordb.similarity_search_with_relevance_scores("What is YOLO?", k=1)

[(Document(metadata={'source': '/content/1506.02640v5.pdf'}, page_content='Detection In The Wild  Academic datasets for object detection draw the training and testing data from the same distribution. In real-world applications it is hard to predict all possible use cases and  YOLO is a fast, accurate object detector, making it ideal for computer vision applications. We connect YOLO to a webcam and verify that it maintains real-time performance,  \x0cVOC 2007 AP 59.2 54.2 43.2 36.5 -  Picasso AP Best F1 0.590 53.3 0.226 10.4 0.458 37.8 0.271 17.8 0.051 1.9  People-Art AP 45 26 32  YOLO R-CNN DPM Poselets [2] D&T [4]  (a) Picasso Dataset precision-recall curves.  (b) Quantitative results on the VOC 2007, Picasso, and People-Art Datasets. The Picasso Dataset evaluates on both AP and best F1 score.  Figure 5: Generalization results on Picasso and People-Art datasets.  Figure 6: Qualitative Results. YOLO running on sample artwork and natural images from the internet. It is mostly accurate a

## Chain Setup

### 🤖 Cell 13 — Initialize the LLM

Creates the chat model that will both rewrite follow-up questions (history-aware retrieval) and generate final answers. `temperature=0` makes its outputs deterministic (no randomness) — useful for reproducible testing.

In [7]:
# Initialize an LLM instance using OpenAI's "gpt-4o-mini" model
# The temperature parameter controls randomness in the generated responses; 0 ensures deterministic outputs
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

### 🕘 Cell 14 — Build a history-aware retriever

This is the key piece that gives the retriever "memory": before searching the vector store, it first asks the LLM to **rewrite** the latest question into a standalone one, using the chat history for context.

- `prompt_str` → instructs the LLM: given the chat history and a possibly-context-dependent question, reformulate it into a standalone question — but do **not** answer it yet, just rewrite it (or return it unchanged if it's already standalone).
- `MessagesPlaceholder(variable_name="chat_history")` → a slot in the prompt template where the actual list of past messages gets inserted at call time, sitting between the system instructions and the new human question.
- `create_history_aware_retriever(llm, vectordb.as_retriever(), prompt_history_aware)` → wires this together into a retriever that accepts `{"input": ..., "chat_history": ...}` and internally: (1) uses the LLM + prompt to produce a standalone question, then (2) runs that standalone question through `vectordb`'s normal retriever.

**Example:** if `chat_history` already contains "What is YOLO?" / "YOLO is a real-time object detection system...", and the next `input` is "What tasks can it be used for?", this step rewrites that into something like "What tasks can YOLO be used for?" *before* searching — so the retriever isn't blindly searching for the ambiguous word "it".

In [8]:
# Define a prompt template for generating answers based on a given context and question
prompt_str = """Given a chat history and the latest user question which might reference context in the chat history,
formulate a standalone question which can be understood without the chat history. Do NOT answer the question,
just reformulate it if needed and otherwise return it as is.
"""

# Use a prompt that includes a MessagesPlaceholder variable under the name "chat_history".
# This allows us to pass in a list of Messages to the prompt using the "chat_history" input key,
# and these messages will be inserted after the system message and before the human message containing the
# latest question.

prompt_history_aware = ChatPromptTemplate.from_messages([
    ("system", prompt_str),                                  # the rewriting instructions above
    MessagesPlaceholder(variable_name="chat_history"),        # slot: past conversation messages get inserted here
    ("human", "{input}"),                                     # slot: the latest raw user question
])

# create_history_aware_retriever constructs a chain that accepts keys input and chat_history as input, and has the same output schema as a retriever
history_aware_retriever = create_history_aware_retriever(
    llm, vectordb.as_retriever(), prompt_history_aware   # LLM to rewrite the question, base retriever to search with, prompt to use for rewriting
)

### 🔗 Cell 16 — Build the final RAG chain

`create_retrieval_chain` glues the history-aware retriever together with an answer-generation step, running them in sequence and keeping track of intermediate outputs.

- `system_prompt` → tells the LLM to answer using only the retrieved context, admit when it doesn't know, and keep answers short (max 3 sentences). `{context}` is a placeholder the chain fills in with the retrieved document text.
- `question_answer_chain = create_stuff_documents_chain(llm, prompt)` → the "answer" half: takes retrieved documents + the question, stuffs the documents directly into the prompt's `{context}` slot, and asks the LLM to answer.
- `rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)` → the full pipeline: rewrite question with history → retrieve relevant chunks → answer using those chunks. Its output includes `input`, `chat_history`, `context` (the retrieved documents), and `answer`.

In [9]:
system_prompt = """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question. If you don't know the answer,
say that you don't know. Use three sentences maximum and keep theanswer concise.

{context}
"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),   # answering instructions + {context} placeholder for retrieved documents
        ("human", "{input}"),         # the (possibly rewritten) standalone question
    ]
)

# create_retrieval_chain combines history aware retriever and the qa chain
question_answer_chain = create_stuff_documents_chain(llm, prompt)              # the "answer using retrieved context" step
rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)  # retrieve -> answer, end to end

### 💬 Cell 17 — First turn: ask a question and manually record it in chat history

Demonstrates the *manual* way of maintaining memory (before we automate it in Cell 21): a plain Python list, `chat_history`, starts empty. We invoke `rag_chain` with the question and the current (empty) history, then append both the `HumanMessage` (the question) and `AIMessage` (the answer) onto `chat_history` ourselves so the next turn can use it.

In [10]:
# Load a empty chat_history list
chat_history = []

# Invoking the chain
question = "What is YOLO?"
response = rag_chain.invoke({"input": question, "chat_history": chat_history})  # chat_history is empty on this first turn

# Appeding question and response answers
chat_history.extend(
    [
        HumanMessage(content=question),          # record what the user asked
        AIMessage(content=response["answer"]),   # record what the assistant answered
    ]
)

### 👀 Cell 18 — Inspect the chat history

Just prints out `chat_history` to confirm it now holds the one `HumanMessage`/`AIMessage` pair from the first turn.

In [11]:
#check chat_history
chat_history

[HumanMessage(content='What is YOLO?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='YOLO, which stands for "You Only Look Once," is a unified model for real-time object detection that frames the task as a regression problem. It uses a single convolutional neural network to predict multiple bounding boxes and class probabilities directly from full images in one evaluation. YOLO is known for its speed and accuracy, processing images at up to 155 frames per second while achieving high mean average precision.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

### 💬 Cell 19 — Second turn: a follow-up question that relies on memory

Asks *"What are the tasks YOLO can be used in?"* — note this doesn't repeat the word "YOLO" being previously established, but more importantly, this time `chat_history` (from Cell 17) is passed in non-empty, so `history_aware_retriever` can use it if a question ever needs disambiguating context.

In [12]:
second_question = "What are the tasks YOLO can be used in?"
response = rag_chain.invoke({"input": second_question, "chat_history": chat_history})  # chat_history now has turn 1 in it

print(response["answer"])

YOLO can be used in various computer vision applications, including real-time object detection in video streams, autonomous driving, and assistive devices that provide scene information. Its fast and accurate detection capabilities make it suitable for tasks that require quick responses, such as tracking moving objects. Additionally, YOLO can generalize well to different domains, making it applicable in areas like artwork recognition and surveillance.


### 🔁 Automating chat history: from a manual list to per-session memory

So far, *we* were responsible for creating `chat_history` and appending to it after every turn — tedious and error-prone in a real app with multiple users. The next step automates this: each user gets a unique `session_id`, and `RunnableWithMessageHistory` automatically loads that session's history before each call and saves the new turn after, with no manual list-appending needed.

### 🗃️ Cell 21 — Wire up automatic, per-session chat history

- `store = {}` → a simple in-memory dictionary mapping `session_id → ChatMessageHistory` (in a real app this would be a database instead).
- `get_session_history(session_id)` → looks up (or creates, if new) the `ChatMessageHistory` for a given session id.
- `RunnableWithMessageHistory(rag_chain, get_session_history, ...)` → wraps `rag_chain` so that on every `.invoke(...)` call, it: (1) fetches the right session's history via `get_session_history`, (2) injects it into the chain's `chat_history` input, (3) runs the chain, and (4) saves the new question/answer back into that session's history — all automatically.
  - `input_messages_key="input"` → which input field holds the new user message.
  - `history_messages_key="chat_history"` → which input field the past-messages history should be injected into.
  - `output_messages_key="answer"` → which field of the chain's output is the actual answer to record into history.

In [13]:
store = {}  # session_id -> ChatMessageHistory, held in memory for this notebook run

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()  # first time we've seen this session: start a fresh history
    return store[session_id]

conversational_rag_chain = RunnableWithMessageHistory(
    rag_chain,                            # the chain to wrap with automatic history management
    get_session_history,                  # how to fetch/create a session's history
    input_messages_key="input",           # which input field is the new user message
    history_messages_key="chat_history",  # which input field the past messages get injected into
    output_messages_key="answer",         # which output field is the answer to record into history
)

C:\Users\Avado\AppData\Local\Temp\claude\C--Users-Avado-Desktop-LearningGenAI\37e7fada-b4a3-4f8e-b68e-c30c95841747\scratchpad\notebook_run\venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


### 💬 Cell 22 — First automated-memory turn

Invokes `conversational_rag_chain` for `session_id="User_1"` — no manual `chat_history` list needed anymore; it's created and tracked automatically behind the scenes for that session id. `["answer"]` just pulls out the answer string from the full response dict.

In [14]:
conversational_rag_chain.invoke(
    {"input": "What is Transformers?"},
    config={
        "configurable": {"session_id": "User_1"}  # ties this call to User_1's session history
    },
)["answer"]

'Transformers are a type of neural network architecture designed for sequence transduction tasks, such as language translation, that rely entirely on self-attention mechanisms instead of recurrent or convolutional layers. This architecture allows for greater parallelization and efficiency in training, achieving state-of-the-art results in various natural language processing tasks. The Transformer model consists of an encoder-decoder structure, where the encoder processes input sequences and the decoder generates output sequences.'

### 💬 Cell 23 — Second automated-memory turn, same session

Asks a follow-up ("Transformers vs YOLO") in the *same* `session_id="User_1"`, so the chain automatically has access to the previous turn's Q&A when rewriting/answering this one. This time the full response dict is returned (not just `["answer"]`), so you can see `input`, `chat_history`, `context`, and `answer` all together.

In [15]:
# This will output input, chat_history, contexts and answer
conversational_rag_chain.invoke(
    {"input": "Transformers vs YOLO"},
    config={
        "configurable": {"session_id": "User_1"}  # same session as Cell 22, so its history carries over
    },
)

{'input': 'Transformers vs YOLO',
 'chat_history': [HumanMessage(content='What is Transformers?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Transformers are a type of neural network architecture designed for sequence transduction tasks, such as language translation, that rely entirely on self-attention mechanisms instead of recurrent or convolutional layers. This architecture allows for greater parallelization and efficiency in training, achieving state-of-the-art results in various natural language processing tasks. The Transformer model consists of an encoder-decoder structure, where the encoder processes input sequences and the decoder generates output sequences.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])],
 'context': [Document(metadata={'source': '/content/1506.02640v5.pdf'}, page_content='Detection In The Wild  Academic datasets for object detection draw the training and testing data from the same distribution. In r

### 📜 Cell 24 — Inspect the full stored conversation

Loops through `store["User_1"].messages` (everything `RunnableWithMessageHistory` has automatically recorded for this session across both calls) and prints each message with an "AI" or "User" label, confirming the automatic memory captured the whole conversation correctly.

In [16]:
# Check the chat_history of the store dict
for message in store["User_1"].messages:
    if isinstance(message, AIMessage):
        prefix = "AI"     # label assistant turns
    else:
        prefix = "User"   # label human turns

    print(f"{prefix}: {message.content}\n")

User: What is Transformers?

AI: Transformers are a type of neural network architecture designed for sequence transduction tasks, such as language translation, that rely entirely on self-attention mechanisms instead of recurrent or convolutional layers. This architecture allows for greater parallelization and efficiency in training, achieving state-of-the-art results in various natural language processing tasks. The Transformer model consists of an encoder-decoder structure, where the encoder processes input sequences and the decoder generates output sequences.

User: Transformers vs YOLO

AI: Transformers and YOLO serve different purposes in the field of machine learning. Transformers are primarily used for natural language processing tasks, leveraging self-attention mechanisms to understand context in sequences, while YOLO (You Only Look Once) is a real-time object detection system that frames detection as a regression problem to predict bounding boxes and class probabilities directl